In [ ]:
import asyncio
import csv
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
import nest_asyncio
from pathlib import Path
import ast

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Import your existing conversation generator
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator


class ConversationPairGenerator:
    """Generator for movie recommendation conversations based on CSV data with dual sampling and rewards"""
    
    def __init__(self, base_config: ConversationConfig, user_prompt_template_path: str, 
                 terminal_signal: str = "[[TERMINATE CHAT]]"):
        self.base_config = base_config
        self.terminal_signal = terminal_signal
        
        # Load the user meta prompt template
        with open(user_prompt_template_path, 'r') as f:
            self.user_prompt_template = f.read()
    
    def load_csv_data(self, csv_path: str) -> pd.DataFrame:
        """Load and parse the CSV data"""
        df = pd.read_csv(csv_path)
        
        # Parse the conversation column (assuming it's stored as string representation of list)
        def parse_conversation(conv_str):
            try:
                # Handle the conversation string - it might be a JSON string or Python literal
                if isinstance(conv_str, str):
                    return ast.literal_eval(conv_str)
                return conv_str
            except (ValueError, SyntaxError) as e:
                logger.warning(f"Failed to parse conversation: {e}")
                return []
        
        df['conversation_parsed'] = df['conversation'].apply(parse_conversation)
        return df
    
    async def generate_conversations_batch_with_rewards(self, df: pd.DataFrame, init_prompts: List=None, 
                                                       samples_per_conversation: int = 2) -> List[Dict]:
        """
        Generate multiple conversations per input with parallel reward generation
        
        Args:
            df: DataFrame with conversation data
            init_prompts: Initial prompts for conversations
            samples_per_conversation: Number of conversation samples to generate per input
        """
        
        total_rows = len(df)
        total_conversations = total_rows * samples_per_conversation
        logger.info(f"Starting batch generation for {total_conversations} conversations ({samples_per_conversation} samples per input)")
        
        # Create expanded configs - duplicate each row for multiple samples
        batch_configs = []
        expanded_init_prompts = []
        
        if not init_prompts:
            init_prompts = [None] * total_rows
        
        for idx, row in df.iterrows():
            # Create conversation text
            conv_text = ""
            for msg in row['conversation_parsed']:
                role = msg['role']
                content = msg['content']
                if role == 'user':
                    conv_text += f"User: {content}\n"
                elif role == 'assistant':
                    conv_text += f"Assistant: {content}\n"
            
            # Fill in the template
            custom_user_prompt = self.user_prompt_template.format(
                conversation=conv_text.strip(),
                ground_truth=row['ground_truth'],
                terminal_signal=self.terminal_signal,
                chat_history="{chat_history}"  # Keep this placeholder for UserSimulator
            )
            
            # Create multiple samples for this conversation
            for sample_idx in range(samples_per_conversation):
                config = {
                    'user_meta_prompt': custom_user_prompt,
                    'user_generation_kwargs': self.base_config.user_generation_kwargs,
                    'initial_prompt': "I'm looking for a movie recommendation.",
                    # Add metadata
                    'dialog_id': row['dialog_id'],
                    'sample_id': sample_idx,  # Track which sample this is
                    'ground_truth': row['ground_truth'],
                    'original_conversation': row['conversation_parsed']
                }
                
                batch_configs.append(config)
                expanded_init_prompts.append(init_prompts[idx] if idx < len(init_prompts) else None)
        
        logger.info(f"📝 Prepared {len(batch_configs)} conversation configs ({samples_per_conversation} samples each)")

        # Create a SINGLE generator for batch processing
        logger.info("🔧 Creating single generator instance...")
        generator = MultiTurnConversationGenerator(self.base_config)
        
        # Generate all conversations
        start_time = time.time()
        logger.info("🎬 Starting conversation generation...")
        
        generated_conversations = await generator.generate_conversations_batch(
            prompts=expanded_init_prompts,
            conv_num=total_conversations,
            batch_configs=batch_configs
        )
        
        conversation_time = time.time()
        logger.info(f"✅ Conversation generation completed in {conversation_time - start_time:.2f} seconds")
        
        # Group results by dialog_id to create pairs
        logger.info("📊 Creating conversation pairs...")
        grouped_results = {}
        
        for config, generated_conv in zip(batch_configs, generated_conversations):
            dialog_id = config['dialog_id']
            sample_id = config['sample_id']
            
            if dialog_id not in grouped_results:
                grouped_results[dialog_id] = {
                    'id': dialog_id,
                    'ground_truth': config['ground_truth'],
                    'original_conversation': config['original_conversation'],
                    'conversations': {},
                    'status': {}
                }
            
            # Store conversation by sample_id
            if generated_conv and len(generated_conv) > 0:
                grouped_results[dialog_id]['conversations'][sample_id] = generated_conv
                grouped_results[dialog_id]['status'][sample_id] = 'success'
            else:
                grouped_results[dialog_id]['conversations'][sample_id] = None
                grouped_results[dialog_id]['status'][sample_id] = 'failed'
        
        # Create final pairs
        conversation_pairs = []
        for dialog_id, data in grouped_results.items():
            # Ensure we have both conversations (sample 0 and 1)
            first_conv = data['conversations'].get(0, None)
            second_conv = data['conversations'].get(1, None)
            first_status = data['status'].get(0, 'failed')
            second_status = data['status'].get(1, 'failed')
            
            pair = {
                'id': dialog_id,
                'ground_truth': data['ground_truth'],
                'original_conversation': data['original_conversation'],
                'first_conversation': first_conv,
                'second_conversation': second_conv,
                'first_status': first_status,
                'second_status': second_status,
                'both_successful': first_status == 'success' and second_status == 'success'
            }
            
            conversation_pairs.append(pair)
        
        end_time = time.time()
        successful = sum(1 for r in results if r['status'] == 'success')
        logger.info(f"🎉 Complete pipeline finished in {end_time - start_time:.2f} seconds")
        logger.info(f"📊 Results: {successful}/{total_conversations} successful conversations")
        
        return conversation_pairs
    
    def save_conversation_pairs_to_csv(self, conversation_pairs: List[Dict], output_path: str):
        """Save conversation pairs to CSV file in the specified format"""
        # Create output directory if it doesn't exist
        output_dir = Path(output_path).parent
        output_dir.mkdir(parents=True, exist_ok=True)

        # Prepare data for CSV
        csv_data = []
        for pair in conversation_pairs:
            # Convert conversations to JSON strings
            original_conv_str = json.dumps(pair['original_conversation'])
            first_conv_str = json.dumps(pair['first_conversation']) if pair['first_conversation'] else None
            second_conv_str = json.dumps(pair['second_conversation']) if pair['second_conversation'] else None
            
            csv_row = {
                'id': pair['id'],
                'ground_truth': pair['ground_truth'],
                'original_conversation': original_conv_str,
                'first_conversation': first_conv_str,
                'second_conversation': second_conv_str,
                'first_status': pair['first_status'],
                'second_status': pair['second_status'],
                'both_successful': pair['both_successful']
            }
            
            csv_data.append(csv_row)
        
        # Write to CSV
        df_output = pd.DataFrame(csv_data)
        df_output.to_csv(output_path, index=False)
        logger.info(f"Conversation pairs saved to {output_path}")
        
        # Print summary statistics
        total_pairs = len(conversation_pairs)
        successful_pairs = sum(1 for p in conversation_pairs if p['both_successful'])
        first_success = sum(1 for p in conversation_pairs if p['first_status'] == 'success')
        second_success = sum(1 for p in conversation_pairs if p['second_status'] == 'success')
        
        logger.info(f"Generation Summary:")
        logger.info(f"  Total pairs: {total_pairs}")
        logger.info(f"  Both successful: {successful_pairs} ({successful_pairs/total_pairs:.1%})")
        logger.info(f"  First conversation success: {first_success} ({first_success/total_pairs:.1%})")
        logger.info(f"  Second conversation success: {second_success} ({second_success/total_pairs:.1%})")


async def main_generation():
    """Main function to run conversation pair generation (Step 1)"""
    
    dataset = "redial"
    local_model_path_name = "test_epoch2_seed3"
    alg = "vanilla"
    model = 'llama3-2-1b-instruct'
    samples_per_conversation = 2  # Generate 2 conversations per input
    
    # File paths
    csv_input_path = "../datasets/"+dataset+"/multiturn_form/test.csv"
    user_prompt_template_path = "../prompts/test_user_prompt.txt"
    conversation_pairs_output_path = "multiturn_test/"+alg+"/llama3_2_1B/"+dataset+"/conversation_pairs_generated.csv"
    
    # Create base configuration
    config = ConversationConfig(
        assistant_meta_prompt="You are a helpful movie recommendation assistant. Provide personalized movie suggestions based on user preferences and engage in natural conversation about movies.",
        # This will be overridden for each conversation with custom context
        user_meta_prompt="You are a user looking for movie recommendations. Respond naturally based on the conversation context.",
        max_total_turns=10,
        max_gen_workers=20,
        local_model_path="/home/sagemaker-user/csbai/multiturn_rl/outputs/"+alg+"/"+dataset+"/"+local_model_path_name,
        base_model_path="meta-llama/Llama-3.2-1B-Instruct",
        assistant_generation_kwargs={
            "temperature": 0.7,
            "max_tokens": 512,
            "model": "us.meta.llama3-2-1b-instruct-v1:0",
            "num_retries": 50
        },
        user_generation_kwargs={
            "model": "us.anthropic.claude-sonnet-4-20250514-v1:0",
            "temperature": 0.8,
            "max_tokens": 256,
            "num_retries": 50
        },
        enable_batching=True,
        use_bedrock_assistant= (alg == "vanilla")
    )
    
    # Initialize generator
    pair_generator = ConversationPairGenerator(
        base_config=config,
        user_prompt_template_path=user_prompt_template_path,
        terminal_signal="[[TERMINATE CHAT]]"
    )
    
    # Load data
    logger.info("Loading CSV data...")
    df = pair_generator.load_csv_data(csv_input_path)
    logger.info(f"Loaded {len(df)} conversations from CSV")
    
    # Generate conversation pairs
    logger.info(f"Starting conversation pair generation...")
    start_time = time.time()
    
    conversation_pairs = await pair_generator.generate_conversation_pairs(
        df, 
        samples_per_conversation=samples_per_conversation
    )
    
    end_time = time.time()
    logger.info(f"Total generation time: {end_time - start_time:.2f} seconds")
    
    # Save conversation pairs
    logger.info("Saving conversation pairs...")
    pair_generator.save_conversation_pairs_to_csv(conversation_pairs, conversation_pairs_output_path)
    
    logger.info("Generation step complete!")
    
    return conversation_pairs

# Run generation step
print("🚀 Starting conversation pair generation (Step 1)...")
conversation_pairs = await main_generation()